[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-value_extraction/examples/value_extraction/value_extraction.ipynb)

# Pick a span the regex already found

Phones, amounts and dates have a shape a regular expression can list. Which of the listed spans is "the amount to refund" is a judgement. An LLM asked to copy the value often paraphrases it. Here the regex proposes the candidates, FRIDA-Decisions chooses one of them (or none), and code normalises the chosen span. The value that leaves the function is a substring of the text, not a generated one.

There is always a `none` option, so a list of spans is not forced into an answer.

Pattern: [Pre-parsed value extraction](https://docs.typesafe.ai/cookbooks/pre_parsed_value_extraction_cookbook). The numbers below are fake.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
import re

PATTERNS = {
    "phone": re.compile(r"(?:\+7|8)[\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}"),
    "money": re.compile(r"\d[\d\s]*\d\s*(?:руб(?:лей|ля|\.)?|₽)"),
    "date": re.compile(r"\d{1,2}\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|"
                       r"сентября|октября|ноября|декабря)(?:\s+\d{4})?"),
}
ASK = {
    "phone": "Какая из строк — телефон, на который просят писать?",
    "money": "Какая из строк — сумма к возврату?",
    "date": "Какая из строк — дата, когда привезут заказ?",
}


def spans(text: str, kind: str) -> list[str]:
    return list(dict.fromkeys(match.group(0).strip() for match in PATTERNS[kind].finditer(text)))


def normalize(kind: str, span: str) -> str:
    if kind == "phone":
        digits = re.sub(r"\D", "", span)
        if digits.startswith("8"):
            digits = "7" + digits[1:]
        return digits
    if kind == "money":
        return str(int(re.sub(r"\D", "", span)))
    return re.sub(r"\s+", " ", span).strip()


def request_for(text: str, kind: str, found: list[str]) -> dict:
    criteria = {f"c{i}": span for i, span in enumerate(found)}
    criteria["none"] = "ни одна строка не отвечает на вопрос"
    return {
        "state": text,
        "questions": {"pick": {"type": "choice", "instructions": ASK[kind], "criteria": criteria}},
    }


def extract(judge, text: str, kind: str) -> dict:
    found = spans(text, kind)
    if not found:
        return {"value": "none", "choice": "none", "confidence": None}
    response = judge(request_for(text, kind, found))
    answer = response["answers"]["pick"]
    choice = answer["choice"]
    if choice == "none":
        return {"value": "none", "choice": choice, "confidence": answer["confidence"]}
    index = int(choice[1:])
    return {"value": normalize(kind, found[index]), "choice": choice,
            "confidence": answer["confidence"], "span": found[index]}


## The model points, the regex owns the characters

`value` is the normalised form of a span that occurs in the text, or `none`.


In [4]:
def run(judge, rows: list[dict]) -> None:
    hits = 0
    for row in rows:
        result = extract(judge, row["text"], row["kind"])
        hits += result["value"] == row["value"]
        conf = "n/a" if result["confidence"] is None else f"{result['confidence']:.2f}"
        print(f"{'ok' if result['value'] == row['value'] else 'miss':4} "
              f"got {result['value']:16} gold {row['value']:16} p={conf}  {row['kind']}")
    print(f"\n{hits}/{len(rows)} normalised values match")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-value-extraction/examples/value_extraction/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


ok   got 500              gold 500              p=0.71  money
ok   got 2490             gold 2490             p=0.94  money
ok   got 79160000000      gold 79160000000      p=0.97  phone
ok   got 79990001122      gold 79990001122      p=0.83  phone
ok   got 15 марта         gold 15 марта         p=0.42  date
ok   got 3 апреля 2026    gold 3 апреля 2026    p=0.43  date
ok   got none             gold none             p=n/a  money
ok   got none             gold none             p=n/a  phone

8/8 normalised values match
